# Step by step: does written reasoning fix the one-sentence rule? (preregistered: `docs/preregistration_cot_axis.md`)

The decisive "which is farther east / north, C or R?" questions (and controls), asked twice to the same model: with a
one-word answer, and with "think it through step by step, then end with 'Answer: <name>'". Three models: Llama 3.1
8B, Qwen3-8B and Gemma 3 12B (8-bit weights on an L4, full precision on a bigger GPU).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. Runtime → Change runtime type → **L4** (or A100 / H100). Then **Run all**. About 30–60 minutes each for Llama and
   Qwen and 2–3 hours for Gemma on an L4 (it writes long replies); much less on an A100.

Replies are saved to Drive after every batch; after a disconnect, just press Run all again (finished work is
skipped). The last cell downloads `cot_axis_results.zip`; send that file back.

In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 / A100 / H100"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 20, f"Only {GB:.0f} GB of GPU memory: pick an L4, A100 or H100"
BIG = GB > 35
GEMMA = "gemma3-12b" if BIG else "gemma3-12b-8bit"   # preregistered: bf16 on >= 35 GB, else 8-bit weights
BS_8B, BS_GEMMA = (32, 32) if BIG else (16, 8)        # batch sizes: memory only
print(f"{GB:.0f} GB: Llama and Qwen in bf16; Gemma 3 12B as {GEMMA}")

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/cot_axis.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/cot", exist_ok=True)
print("already saved:", sorted(os.listdir(f"{RES}/cot")))

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' imports); bitsandbytes for 8-bit Gemma
!pip -q install "transformers==4.57.3" nnsight accelerate bitsandbytes
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional); is the gated Llama 3.1 available to this account?
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami, hf_hub_download
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
try:
    hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=os.environ["HF_TOKEN"])
    RUN_LLAMA = True
except Exception:
    RUN_LLAMA = False
    print("Llama 3.1 8B: NO ACCESS; it will be skipped.")
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Step by step] Colab run started (or resumed)")

In [ ]:
# 5. Llama 3.1 8B
if RUN_LLAMA:
    !cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/cot_axis.py --model llama3.1-8b --batch-size {BS_8B} 2>&1 | grep -v -i warning

In [ ]:
# 6. Qwen3-8B
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/cot_axis.py --model qwen3-8b --batch-size {BS_8B} 2>&1 | grep -v -i warning

In [ ]:
# 7. Gemma 3 12B (the paper's main model; longest run)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/cot_axis.py --model {GEMMA} --batch-size {BS_GEMMA} 2>&1 | grep -v -i warning

In [ ]:
# 8. One zip of the results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/cot_axis_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/cot/*"):
        z.write(f, "results/cot/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/cot_axis_results.zip", f"{D}/cot_axis_results.zip")
notify("[Step by step] zip saved: MyDrive/geomap/cot_axis_results.zip")
from google.colab import files
files.download("/content/cot_axis_results.zip")